# TerraMind Sen1Floods11 Training Notebook

This notebook provides an interactive workflow for training TerraMind on the Sen1Floods11 dataset with optional Phisat-2 augmentation. Each cell represents a distinct step in the training pipeline.

## 1. Import Required Libraries

Import all necessary libraries including Lightning, TerraTorch, and related dependencies.

In [ ]:
import logging
from pathlib import Path
from typing import Dict, Any

from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from lightning.pytorch.loggers import TensorBoardLogger

from terratorch.datamodules.sen1floods11 import Sen1Floods11NonGeoDataModule
from terratorch.tasks import SemanticSegmentationTask
from terratorch import BACKBONE_REGISTRY

from data_simulation.phisat2_constants import S2_BANDS_NAMES, S2_BANDS
from data_simulation.phisat2_albumentations import create_phisat2_transform

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

/shared/home/elucas/terra-sat-drift/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/shared/home/elucas/terra-sat-drift/terra_sat_drift/data_simulation/phisat2_constants.py:38: FutureWarning: The geopandas.dataset module is deprecated and will be removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.
  WORLD_GDF = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))


✓ All libraries imported successfully!


## 2. Define Backbone Configuration & Trainer Class

Define backbone indices mapping and the TerraMindSen1FloodsTrainer orchestrator class.

In [ ]:
# Backbone configuration
BACKBONE_SIZES = {"tiny", "small", "base", "large"}
BACKBONE_NECK_INDICES = {
    "tiny": [1, 3, 4, 5],
    "small": [1, 3, 4, 5],
    "base": [2, 5, 8, 11],
    "large": [5, 11, 17, 23],
}


class TerraMindSen1FloodsTrainer:
    """Orchestrator for TerraMind fine-tuning on Sen1Floods11 with staged training."""

    def __init__(
        self,
        backbone_size: str = "small",
        learning_rate: float = 1e-4,
        batch_size: int = 16,
        max_epochs: int = 50,
        output_dir: str = "./outputs",
        num_workers: int = 4,
        use_amp: bool = True,
    ):
        """Initialize trainer."""
        self.backbone_size = backbone_size
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.max_epochs = max_epochs
        self.output_dir = Path(output_dir)
        self.num_workers = num_workers
        self.use_amp = use_amp
        self.output_dir.mkdir(parents=True, exist_ok=True)

    def create_datamodule(
        self,
        root_dir: str,
        phisat_config: Dict[str, Any] | None = None,
    ) -> Sen1Floods11NonGeoDataModule:
        """Create Sen1Floods11 datamodule with optional Phisat-2 augmentation."""
        train_transform = None
        if phisat_config:
            train_transform = create_phisat2_transform(phisat_config)
            
        return Sen1Floods11NonGeoDataModule(
            data_root=root_dir,
            bands=S2_BANDS_NAMES,
            num_workers=self.num_workers,
            batch_size=self.batch_size,
            download=False,
            use_metadata=True
        )

    def create_model(
        self,
        num_classes: int = 2,
        pretrained: bool = True,
    ) -> SemanticSegmentationTask:
        """Create SemanticSegmentationTask with TerraMind backbone."""
        backbone_name = f"terramind_v1_{self.backbone_size}"
        neck_indices = BACKBONE_NECK_INDICES[self.backbone_size]

        # Build TerraMind backbone with pretrained weights
        terramind_backbone = BACKBONE_REGISTRY.build(
            backbone_name,
            pretrained=True,
            modalities=["S2L1C"],
            bands={"S2L1C": S2_BANDS},
        )
        
        model_args = {
            "backbone": terramind_backbone,
            "backbone_pretrained": True,
            "backbone_modalities": ["S2L1C"],
            "backbone_bands": {"S2L1C": S2_BANDS},
            "decoder": "UNetDecoder",
            "decoder_channels": [256, 128, 64, 32],
            "necks": [
                {"name": "SelectIndices", "indices": neck_indices},
                {"name": "ReshapeTokensToImage", "remove_cls_token": False},
                {"name": "LearnedInterpolateToPyramidal"},
            ],
            "num_classes": num_classes,
        }
        
        return SemanticSegmentationTask(
            model_factory="EncoderDecoderFactory",
            model_args=model_args,
            lr=self.learning_rate,
            ignore_index=-1,
        )

    def train(
        self,
        datamodule: Sen1Floods11NonGeoDataModule,
        model: SemanticSegmentationTask,
        stage_1_epochs: int = 10,
        disable_stage_2: bool = True,
    ) -> SemanticSegmentationTask:
        """Execute staged fine-tuning: Stage 1 (frozen) and Stage 2 (full fine-tune)."""
        logger.info("Starting staged fine-tuning on Sen1Floods11")

        # Stage 1: Train only head/decoder (backbone frozen)
        logger.info(f"Stage 1: Training head (epochs 0-{stage_1_epochs})")
        self._freeze_backbone(model)
                
        trainer_stage1 = self._create_trainer(stage=1, max_epochs=stage_1_epochs)
        trainer_stage1.fit(model, datamodule=datamodule)
        
        if not disable_stage_2:
            # Stage 2: Fine-tune entire model (backbone unfrozen)
            logger.info(f"Stage 2: Fine-tuning full model (epochs {stage_1_epochs}-{self.max_epochs})")
            self._unfreeze_backbone(model)
            
            # Reduce learning rate for fine-tuning
            for param_group in model.optimizer.param_groups:
                param_group["lr"] = self.learning_rate * 0.1
            
            trainer_stage2 = self._create_trainer(stage=2, max_epochs=self.max_epochs - stage_1_epochs)
            trainer_stage2.fit(model, datamodule=datamodule, ckpt_path="last")

        return model

    def _freeze_backbone(self, model: SemanticSegmentationTask) -> None:
        """Freeze all backbone parameters."""
        if hasattr(model, "backbone"):
            for param in model.backbone.parameters():
                param.requires_grad = False
            logger.info("Backbone frozen")

    def _unfreeze_backbone(self, model: SemanticSegmentationTask) -> None:
        """Unfreeze all backbone parameters."""
        if hasattr(model, "backbone"):
            for param in model.backbone.parameters():
                param.requires_grad = True
            logger.info("Backbone unfrozen")

    def _create_trainer(self, stage: int, max_epochs: int) -> Trainer:
        """Create PyTorch Lightning trainer with callbacks."""
        checkpoint_callback = ModelCheckpoint(
            dirpath=self.output_dir / f"checkpoints_stage{stage}",
            filename="model-{epoch:02d}-{val_loss:.3f}",
            monitor="val_loss",
            mode="min",
            save_top_k=3,
            verbose=True,
        )

        early_stopping = EarlyStopping(
            monitor="val_loss",
            patience=5,
            mode="min",
            verbose=True,
        )

        logger_tb = TensorBoardLogger(
            save_dir=self.output_dir,
            name=f"stage{stage}",
            version=0,
        )

        return Trainer(
            max_epochs=max_epochs,
            callbacks=[checkpoint_callback, early_stopping],
            logger=logger_tb,
            log_every_n_steps=10,
            enable_progress_bar=True,
        )

print("✓ TerraMindSen1FloodsTrainer class defined!")

## 3. Define Configuration Parameters

Set up all training hyperparameters and configuration options.

In [ ]:
# Training hyperparameters
BACKBONE_SIZE = "small"  # Options: "tiny", "small", "base", "large"
LEARNING_RATE = 1e-4
BATCH_SIZE = 16
MAX_EPOCHS = 50
NUM_WORKERS = 4
USE_AMP = True

# Dataset paths
DATA_ROOT = "datasets/sen1floods11"
OUTPUT_DIR = "./outputs/terramind_sen1floods"

# Phisat-2 augmentation flags
APPLY_BAND_MISALIGNMENT = False
APPLY_PAN_BAND = False
APPLY_PSF = False
APPLY_SNR = False
PROCESSING_LEVEL = "L1A"

# External executables (optional)
PSF_EXECUTABLE = "./executables/phisat2_unix.bin"
SNR_EXECUTABLE = None

# Training stages
STAGE_1_EPOCHS = 10
DISABLE_STAGE_2 = False

print("✓ Configuration parameters set!")

## 4. Initialize TerraMindSen1FloodsTrainer

Instantiate the trainer with the configured parameters.

In [ ]:
# Instantiate trainer
trainer = TerraMindSen1FloodsTrainer(
    backbone_size=BACKBONE_SIZE,
    learning_rate=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    output_dir=OUTPUT_DIR,
    num_workers=NUM_WORKERS,
    use_amp=USE_AMP,
)

print(f"✓ Trainer initialized!")
print(f"  - Backbone size: {BACKBONE_SIZE}")
print(f"  - Learning rate: {LEARNING_RATE}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Max epochs: {MAX_EPOCHS}")
print(f"  - Output directory: {OUTPUT_DIR}")

## 5. Create Data Module

Create and setup the Sen1Floods11 datamodule with optional Phisat-2 augmentation transforms.

In [ ]:
# Build Phisat-2 configuration
phisat_config = {
    "apply_band_misalignment": APPLY_BAND_MISALIGNMENT,
    "apply_pan_band": APPLY_PAN_BAND,
    "apply_psf": APPLY_PSF,
    "apply_snr": APPLY_SNR,
    "processing_level": PROCESSING_LEVEL,
}

# Add executables if provided
if PSF_EXECUTABLE:
    phisat_config["psf_executable"] = PSF_EXECUTABLE
    logger.info(f"Using PSF executable: {PSF_EXECUTABLE}")

if SNR_EXECUTABLE:
    phisat_config["snr_executable"] = SNR_EXECUTABLE
    logger.info(f"Using SNR executable: {SNR_EXECUTABLE}")

logger.info(f"Phisat-2 config: {phisat_config}")

# Create datamodule
datamodule = trainer.create_datamodule(
    root_dir=DATA_ROOT,
    phisat_config=phisat_config if any(phisat_config.values()) else None,
)
datamodule.setup(stage="fit")

# Display dataset information
print(f"✓ Data module created and setup!")
print(f"  - Train dataset size: {len(datamodule.train_dataset)}")
print(f"  - Val dataset size: {len(datamodule.val_dataset)}")

## 6. Create Model

Build the SemanticSegmentationTask with TerraMind backbone, decoder, and necks configuration.

In [ ]:
# Create model
model = trainer.create_model(num_classes=2, pretrained=True)

print(f"✓ Model created successfully!")
print(f"  - Backbone: terramind_v1_{BACKBONE_SIZE}")
print(f"  - Decoder: UNetDecoder")
print(f"  - Number of output classes: 2")
print(f"  - Learning rate: {LEARNING_RATE}")

## 7. Execute Staged Training

Run the staged fine-tuning process:
- **Stage 1**: Train head/decoder only (backbone frozen)
- **Stage 2**: Fine-tune entire model (backbone unfrozen)

In [ ]:
# Execute staged training
trained_model = trainer.train(
    datamodule=datamodule,
    model=model,
    stage_1_epochs=STAGE_1_EPOCHS,
    disable_stage_2=DISABLE_STAGE_2,
)

print(f"✓ Training completed!")
print(f"  - Stage 1 epochs: {STAGE_1_EPOCHS}")
print(f"  - Stage 2 enabled: {not DISABLE_STAGE_2}")
print(f"  - Total epochs: {MAX_EPOCHS}")

## 8. Save Final Model

Save the trained model checkpoint to the output directory.

In [ ]:
# Save final model
final_ckpt = Path(OUTPUT_DIR) / "final_model.ckpt"
trainer._create_trainer(stage=2, max_epochs=1).save_checkpoint(final_ckpt)

logger.info(f"Training complete! Final model saved to {final_ckpt}")
print(f"✓ Final model checkpoint saved!")
print(f"  - Location: {final_ckpt}")
print(f"\nTraining pipeline finished successfully!")